# Structure of Revelation Segments

This notebook calculates cross-reference counts for the sections used in the Structure of Revelation graphic. The segment definitions and colors live in `../data/structure_of_revelation_schema.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go

cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_DIR = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data").exists():
    PROJECT_DIR = cwd.parent
elif (cwd / "revelation" / "data").exists():
    PROJECT_DIR = cwd / "revelation"
else:
    raise FileNotFoundError("Could not locate the revelation/data directory from the current working directory.")

SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from openbible_data import load_cross_references
from revelation_analysis import (
    add_structure_segment_counts,
    get_rev_from,
    load_structure_schema,
    validate_structure_counts,
)

In [2]:
df = load_cross_references(DATA_DIR / "cross_references_revelation_public.csv")
rev_from = get_rev_from(df)

schema = load_structure_schema(DATA_DIR / "structure_of_revelation_schema.csv")
schema

,Title,Category,Section,Kayser,Range,Whole chapter,Chapter start,Verse start,Chapter end,Verse end,RGB,HTML Color
0,John's Vision of Christ,Prologue,Prologue,Prologue,Rev 1,1,1,1,1,20,"0.85, 0.85, 0.85",LightGray
1,Letters to the Seven Churches,Sevens,Seven Churches,Seven Churches,Rev 2-3,1,2,1,3,22,"0.06, 0.62, 0.84",Cyan
2,Vision of the Throne Room,Interlude,The Throne,Seven Seals,Rev 4,1,4,1,4,11,"0.8, 0.6, 0.0",GoldenRod
3,The Seven Seals,Sevens,The scroll and the first six seals,Seven Seals,Rev 5-6,1,5,1,6,17,"0.0, 0.0, 0.0",Black
4,Vision of Great Multitude,Interlude,"The five angels, the 144,000, the Great Multitude",Seven Seals,Rev 7,1,7,1,7,17,"0.0, 0.0, 0.0",Black
5,The Seventh Seal,Fulfillment,Silence,Seven Seals,Rev 8:1-5,0,8,1,8,5,"0.4, 0.2, 0.6",RebeccaPurple
6,The Seven Trumpets,Sevens,The first six trumpets,Seven Trumpets,Rev 8:6-9:21,0,8,6,9,21,"0.0, 0.0, 0.8039",MediumBlue
7,The Second Woe,Interlude,"The angel and the little scroll, the two witne...",Seven Trumpets,Rev 10-11:14,0,10,1,11,14,"0.0, 0.0, 0.8039",MediumBlue
8,The Seventh Trumpet,Fulfillment,Glory,Seven Trumpets,Rev 11:15-19,0,11,15,11,19,"0.4, 0.2, 0.6",RebeccaPurple
9,Center of the Chiasm,Interlude,The Cosmic War,Seven Visions,Rev 12-15,1,12,1,15,8,"0.0, 0.502, 0.0",Green


## Segment Counts

Counts are calculated directly from each schema row using `Chapter start`, `Verse start`, `Chapter end`, and `Verse end`. This avoids hand-coded segment variables and keeps the schema as the single source of truth.

In [3]:
structure_counts = add_structure_segment_counts(schema, rev_from)
validate_structure_counts(structure_counts, rev_from)

structure_counts[[
    "Range",
    "Title",
    "Category",
    "Section",
    "Kayser",
    "HTML Color",
    "Number of cross-references",
]]

,Range,Title,Category,Section,Kayser,HTML Color,Number of cross-references
0,Rev 1,John's Vision of Christ,Prologue,Prologue,Prologue,LightGray,388
1,Rev 2-3,Letters to the Seven Churches,Sevens,Seven Churches,Seven Churches,Cyan,919
2,Rev 4,Vision of the Throne Room,Interlude,The Throne,Seven Seals,GoldenRod,219
3,Rev 5-6,The Seven Seals,Sevens,The scroll and the first six seals,Seven Seals,Black,443
4,Rev 7,Vision of Great Multitude,Interlude,"The five angels, the 144,000, the Great Multitude",Seven Seals,Black,311
5,Rev 8:1-5,The Seventh Seal,Fulfillment,Silence,Seven Seals,RebeccaPurple,76
6,Rev 8:6-9:21,The Seven Trumpets,Sevens,The first six trumpets,Seven Trumpets,MediumBlue,322
7,Rev 10-11:14,The Second Woe,Interlude,"The angel and the little scroll, the two witne...",Seven Trumpets,MediumBlue,373
8,Rev 11:15-19,The Seventh Trumpet,Fulfillment,Glory,Seven Trumpets,RebeccaPurple,153
9,Rev 12-15,Center of the Chiasm,Interlude,The Cosmic War,Seven Visions,Green,1123


In [4]:
structure_counts["Number of cross-references"].sum(), len(rev_from)

(6495, 6495)

## Cross-References from Each Revelation Section

The bar colors are read from the schema so they stay aligned with the Structure of Revelation graphic.

In [5]:
custom_data = structure_counts[
    ["Range", "Title", "Category", "Section", "Kayser", "Number of cross-references"]
].values

fig = go.Figure(go.Bar(
    x=structure_counts["Number of cross-references"],
    y=structure_counts["Range"],
    orientation="h",
    marker=dict(color=structure_counts["HTML Color"]),
    text=structure_counts["Number of cross-references"],
    textposition="outside",
    customdata=custom_data,
    hovertemplate=(
        "Range: %{customdata[0]}<br>"
        "Title: %{customdata[1]}<br>"
        "Category: %{customdata[2]}<br>"
        "Section: %{customdata[3]}<br>"
        "Kayser: %{customdata[4]}<br>"
        "Number of cross-references: %{customdata[5]}<extra></extra>"
    ),
))

fig.update_layout(
    title=dict(
        text="Cross-References from Each Revelation Section",
        font=dict(size=28),
    ),
    xaxis=dict(
        title=dict(text="number of cross-references", font=dict(size=18)),
    ),
    height=760,
    width=1200,
)
fig.update_yaxes(autorange="reversed", tickfont=dict(size=15))
fig.update_xaxes(tickfont=dict(size=13))
fig.update_traces(textfont=dict(size=15))
fig.show()